# Task 3 — Optimizer Pair Correctness Tests

This notebook walks through **what** the three `@pair` correctness tests do, **why** they exist, and **how to verify them on a GPU**.

The G6 Performance team has two kinds of changes:
- **Speed-up**: do the same computation faster (batched matmuls, fused ops, prefetch)
- **Correctness gate**: prove the fast path gives the same answer as the slow reference path

The `@pair` framework is that correctness gate. Every fast path must register a pair, and every pair must:
1. Pass a relative-error check against an fp64 reference
2. Fail when the output is scaled by 1.01 (proves tolerance is tight enough)
3. Give bit-identical results on two consecutive runs (determinism check)

## 0. Setup

Clone the repo and install dependencies.

In [ ]:
!git clone https://github.com/adinathauti/era-v5-lab.git
%cd era-v5-lab/teams/06-performance
!git checkout g6/optimizer-pairs
!git pull origin g6/optimizer-pairs

In [ ]:
!pip install torch pytest pyyaml numpy tokenizers -q

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if device == 'cuda':
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Native bf16: {torch.cuda.is_bf16_supported()}")

---
## Pair 1 — `opt.muon.batched`

### What is Muon?

Muon is an optimizer for the 2D block weights (attention, MLP). Instead of a plain gradient step it:
1. Applies Nesterov momentum
2. Runs **Newton-Schulz (NS) orthogonalisation** — 5 iterations of a polynomial that pushes the update matrix toward its SVD's U·Vᵀ (approximately orthogonal)
3. Scales by √(max(rows, cols)) so wide and tall matrices get the same effective step size

### The speed-up: `impl="batched"`

| impl | what it does | bottleneck |
|---|---|---|
| `loop` | one `torch.mm` per matrix per NS step | many small kernel launches |
| `batched` | stacks same-shape matrices → one `torch.bmm` per NS step | one bigger kernel launch |

For a model with 18 attention/MLP matrices of the same shape, `batched` replaces 18×5 = 90 `mm` calls with 5 `bmm` calls.

### The pair test

Runs both implementations on the same 3 parameters for 3 steps (with one missing gradient to test the skip logic) and checks the final parameter values agree within `tol=2e-3`.

In [ ]:
from models.optim import Muon

torch.manual_seed(42)
shape = (4, 3)
initial = [torch.randn(shape, device=device) for _ in range(3)]
step_grads = [
    [torch.linspace(-0.3, 0.4, 12).reshape(shape).to(device) * (s + i + 1)
     for i in range(3)]
    for s in range(3)
]
kw = dict(lr=0.025, momentum=0.8, ns_steps=5, weight_decay=0.03)

def run_muon(impl):
    params = [nn.Parameter(p.clone()) for p in initial]
    opt = Muon(params, impl=impl, **kw)
    for s in range(3):
        for i, p in enumerate(params):
            # step 1, param 1: simulate a missing gradient (unused expert path)
            p.grad = None if (s == 1 and i == 1) else step_grads[s][i].clone()
        opt.step()
    return torch.stack([p.detach() for p in params])

loop_out    = run_muon('loop')
batched_out = run_muon('batched')

abs_diff = (batched_out - loop_out).abs()
rel_err  = abs_diff.norm() / loop_out.norm()

print(f"max abs diff : {abs_diff.max():.6e}")
print(f"rel_err      : {rel_err:.6e}")
print(f"tol          : 2e-03")
print(f"PASS         : {rel_err.item() < 2e-3}")

### GPU note

The Newton-Schulz computation internally casts to **bf16** on any GPU with native bf16 support (A100, H100). `torch.bmm` in bf16 accumulates differently from `torch.mm` in bf16, so on an A100 you may see `rel_err` around 1–4e-3 (still below the 1e-2 drill threshold). On a T4 (free Colab) NS falls back to fp32 and the error is ~0.

---
## Pair 2 — `opt.adamw.foreach`

### Context: MuonAdamW (Polar Express)

The base config uses `polar_muon_adamw` — a single combined optimizer that runs Polar Express Muon on 2D block weights and AdamW on everything else (embeddings, norms, biases, head). The AdamW side has two implementations:

| impl | what it does | bottleneck |
|---|---|---|
| `loop` | one parameter at a time: `exp_avg.lerp_()`, `sqrt()`, `addcdiv_()` | many small ops |
| `foreach` | all fp32 params in a group at once via `torch._foreach_*` fused ops | fewer kernel launches |

`_foreach_lerp_`, `_foreach_sqrt_`, `_foreach_addcdiv_` each process the whole list in one CUDA launch instead of N launches.

### The pair test

Uses two 2D params (same group → get batched together by `_foreach`) and one 1D param (separate group, no weight decay). Runs 3 steps and checks final values agree within `tol=1e-6`.

In [ ]:
from models.optim import MuonAdamW

torch.manual_seed(42)
initial_aw = [torch.randn(3, 4, device=device),
              torch.randn(2, 3, device=device),
              torch.randn(5,    device=device)]
adamw_kw = dict(lr=0.001, betas=(0.9, 0.95), eps=1e-8)

def get_grads(step):
    return [
        torch.linspace(-0.1, 0.1, 12).reshape(3, 4).to(device) * (step + 1) * 0.1,
        torch.linspace(0.05, 0.15, 6).reshape(2, 3).to(device) * (step + 2) * 0.1,
        torch.ones(5, device=device) * (step + 3) * 0.01,
    ]

def run_adamw(impl):
    params = [nn.Parameter(p.float().clone()) for p in initial_aw]
    groups = [
        # params[0] and params[1] go in the same group → _foreach batches them together
        {'params': [params[0], params[1]], 'kind': 'adamw',
         'weight_decay': 0.01, 'impl': impl, **adamw_kw},
        # params[2] is 1D (bias-like) → no weight decay
        {'params': [params[2]], 'kind': 'adamw',
         'weight_decay': 0.0,  'impl': impl, **adamw_kw},
    ]
    opt = MuonAdamW(groups)
    for step in range(3):
        for p, g in zip(params, get_grads(step)):
            p.grad = g.float().clone()
        opt.step()
    return torch.cat([p.detach().flatten() for p in params])

loop_aw    = run_adamw('loop')
foreach_aw = run_adamw('foreach')

abs_diff = (foreach_aw - loop_aw).abs()
rel_err  = abs_diff.norm() / loop_aw.norm()

print(f"max abs diff : {abs_diff.max():.6e}")
print(f"rel_err      : {rel_err:.6e}")
print(f"tol          : 1e-06")
print(f"PASS         : {rel_err.item() < 1e-6}")

This should be **exactly 0** (or machine epsilon) on any device. `_foreach_*` ops are element-wise fusions — they don't change accumulation order for per-element ops like `lerp`, `sqrt`, `addcdiv`.

---
## Pair 3 — `data.prefetch`

### What prefetch does

During training the timeline looks like:

```
Without prefetch:  [load batch] → [forward] → [backward] → [load batch] → ...
With prefetch:     [load batch] → [forward + load next batch] → [backward] → [swap] → ...
```

With `prefetch=True`, `loader.prefetch()` is called right after queuing the backward pass. While the GPU is busy computing gradients, the CPU samples the next batch from the shard. By the time the optimizer step finishes, the batch is already pinned in host memory waiting for a non-blocking `.to(device)` copy.

### Why it must produce identical batches

Both paths call `_host_batch()` exactly once per step, advancing the numpy RNG by the same amount in the same order. If they ever disagreed, training runs with and without prefetch would diverge — making the 5% repeat check meaningless.

### The pair test

Runs 4 steps with `prefetch=True` (fast) and `prefetch=False` (ref) from the same seed and checks the token IDs are **exactly identical** (`tol=0.0`).

In [ ]:
import os
from dataio.loader import Loader

batch_size, block_size, n_steps, seed = 2, 8, 4, 17
tokenizer_path = os.environ.get('BRAHMIC_TOKENIZER_PATH')

def run_loader(prefetch):
    loader = Loader('synthetic', batch_size, block_size, 'cpu',
                    seed=seed, prefetch=prefetch, tokenizer_path=tokenizer_path)
    xs, ys = [], []
    for _ in range(n_steps):
        if prefetch:
            loader.prefetch()   # sample next batch while 'GPU is busy'
        x, y = loader.next_batch()
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.stack(ys)

xs_ref,  ys_ref  = run_loader(prefetch=False)
xs_fast, ys_fast = run_loader(prefetch=True)

x_match = torch.equal(xs_ref, xs_fast)
y_match = torch.equal(ys_ref, ys_fast)

print(f"x batches identical: {x_match}")
print(f"y batches identical: {y_match}")
print(f"PASS: {x_match and y_match}")

# Show a sample batch to make it concrete
print(f"\nSample x[0] (first batch, first row, first 8 tokens):")
print(xs_ref[0, 0].tolist())

---
## The 1.01 Drill

For each pair, the framework also verifies that wrapping the fast output with `scaled(1.01)` (multiply every output tensor by 1.01) causes the test to **fail**. This proves the tolerance is tight enough to catch a 1% error — if it weren't, a subtly broken fast implementation could slip through.

We can demonstrate this manually:

In [ ]:
# Demonstrate: 1% error on the muon output exceeds tol=2e-3
scaled_out = batched_out * 1.01
err_drill = (scaled_out - loop_out).norm() / loop_out.norm()
print(f"opt.muon.batched  drill rel_err : {err_drill:.4f}  (should be ~0.01)")
print(f"exceeds tol=2e-3                : {err_drill.item() > 2e-3}  ← drill FAILS as required")
print()

# Same for adamw
scaled_aw = foreach_aw * 1.01
err_drill_aw = (scaled_aw - loop_aw).norm() / loop_aw.norm()
print(f"opt.adamw.foreach drill rel_err : {err_drill_aw:.4f}")
print(f"exceeds tol=1e-6                : {err_drill_aw.item() > 1e-6}  ← drill FAILS as required")
print()

# Same for prefetch (tol=0.0, any nonzero error fails)
xs_ref_f  = xs_ref.float()
xs_fast_f = (xs_fast.float() * 1.01)
err_drill_pf = (xs_fast_f - xs_ref_f).norm() / xs_ref_f.norm()
print(f"data.prefetch     drill rel_err : {err_drill_pf:.4f}")
print(f"exceeds tol=0.0                 : {err_drill_pf.item() > 0.0}   ← drill FAILS as required")

---
## Run the Formal Pytest Suite

This runs the actual registered `@pair` tests through the framework — including the fp64 reference comparison and the determinism check (two runs must be bit-identical).

In [ ]:
!python -m pytest correctness/test_optimizer.py::test_pair -v

In [ ]:
# 1.01 drill via test_framework (must FAIL inside pytest.raises → shows as PASS)
!python -m pytest correctness/test_framework.py::test_every_registered_pair_scaled_fails \
    -k "opt.muon or opt.adamw.foreach or data.prefetch" -v

---
## Summary

| Pair | Fast path | Ref path | Tolerance | GPU concern |
|---|---|---|---|---|
| `opt.muon.batched` | `torch.bmm` on stacked matrices | per-matrix `torch.mm` | 2e-3 | bf16 accumulation on A100 may give ~1–4e-3 error |
| `opt.adamw.foreach` | `_foreach_*` fused ops | sequential per-param ops | 1e-6 | none — fp32 element-wise ops are identical |
| `data.prefetch` | pre-sample while GPU is busy | sample on demand | 0.0 (exact) | none — always CPU, integer RNG |

In [ ]:
# Print the comparison table
PARTS = ("opt:opt.polar_muon_adamw", "opt.muon.ns")

before = extract_parts(before_jsons[0])
after  = extract_parts(after_jsons[0])

print(f"\n{'part':30s}  {'before (loop)':>14s}  {'after (batched)':>16s}  {'speedup':>8s}")
print("-" * 76)
for p in PARTS:
    b = before.get(p)
    a = after.get(p)
    if b is not None and a is not None:
        speedup = b / a
        print(f"{p:30s}  {b:>12.3f} ms  {a:>14.3f} ms  {speedup:>7.2f}x")
    else:
        print(f"{p:30s}  {'N/A':>12s}  {'N/A':>14s}  {'?':>8s}")

In [ ]:
# AFTER: muon_impl=batched, adamw_impl=foreach (base.yaml defaults)  (~10 min)
after_jsons = bench_repeat2("after_batched")

In [ ]:
# BEFORE: muon_impl=loop, adamw_impl=loop  (~10 min)
before_jsons = bench_repeat2("before_loop", "optim.muon_impl=loop optim.adamw_impl=loop")

In [ ]:
import subprocess, json, glob, os

def bench_repeat2(tag, extra_overrides=""):
    """Run perfkit.bench --repeat 2 and return the two run JSON paths."""
    cmd = (
        f"python -m perfkit.bench "
        f"--config configs/base.yaml "
        f"--modes clean,layer "
        f"--level 3 "
        f"--repeat 2 "
        f"--tag {tag} "
        f"data.source=synthetic "
        f"{extra_overrides}"
    )
    print(f"\n{'='*60}")
    print(f"Running: {cmd}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
    # Find the two JSON files written
    jsons = sorted(glob.glob(f"results/runs/{tag}*.json"))
    print(f"Written: {jsons}")
    return jsons

def extract_parts(json_path, parts=("opt:opt.polar_muon_adamw", "opt.muon.ns")):
    """Return {part_name: median_ms} for the requested parts."""
    with open(json_path) as f:
        run = json.load(f)
    out = {}
    for p in run.get("parts", []):
        if p["name"] in parts:
            out[p["name"]] = p.get("median_ms") or p.get("ms")
    return out

---
## Timing: before and after (Task 3 requirement)

Task 3 asks for the time per part for `opt:opt.polar_muon_adamw` and `opt.muon.ns` **before** (loop/loop) and **after** (batched/foreach), using `--repeat 2`.

Two runs each, each in its own process, compared for the 5% repeatability check.  
`data.source=synthetic` avoids the fineweb download on Colab.

Each block takes ~10 minutes (10 warmup + 100 timed steps × 2 processes).